<a href="https://colab.research.google.com/github/yadavgagan01/ddos-ml-detection/blob/main/CMP7239_CVE_ML_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMP7239 Applied Machine Learning – Assessment 2
## CVE Vulnerability Severity Classification using Machine Learning
**Dataset:** NVD Last 30 Days (nvd_last_30days.csv)  
**Task:** Multi-class Classification – Predict CVSS v3 Severity (CRITICAL / HIGH / MEDIUM / LOW)  
**Algorithms:** Random Forest · XGBoost · Logistic Regression · K-Nearest Neighbours  
**Birmingham City University | Module: CMP7239**

---
## 1. Install & Import Libraries

In [1]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, pickle, os, warnings
warnings.filterwarnings('ignore')
from datetime import datetime

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc)

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

plt.rcParams['figure.dpi'] = 120
PALETTE = ['#E74C3C', '#E67E22', '#3498DB', '#2ECC71']
SEVERITY_ORDER = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
print('Libraries imported successfully.')

Libraries imported successfully.


---
## 2. Load Dataset

In [ ]:
try:
    from google.colab import files
    print('Upload nvd_last_30days.csv when prompted')
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
except Exception:
    DATA_PATH = 'nvd_last_30days.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head(3)

Upload nvd_last_30days.csv when prompted


---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Overview ===')
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')
print('\n--- Data Types ---')
print(df_raw.dtypes)
print('\n--- Missing Values ---')
miss = df_raw.isnull().sum()
print(miss[miss > 0])

In [ ]:
df_eda = df_raw.dropna(subset=['CVSS_v3_Severity']).copy()
df_eda = df_eda[df_eda['CVSS_v3_Severity'].isin(SEVERITY_ORDER)]
counts = df_eda['CVSS_v3_Severity'].value_counts().reindex(SEVERITY_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('EDA - CVE Severity Distribution', fontsize=14, fontweight='bold')
bars = axes[0].bar(SEVERITY_ORDER, counts.values, color=PALETTE, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Count per Severity Level')
axes[0].set_ylabel('Number of CVEs')
axes[0].set_xlabel('Severity')
axes[0].grid(axis='y', alpha=0.3)
axes[1].pie(counts.values, labels=SEVERITY_ORDER, autopct='%1.1f%%', colors=PALETTE,
            startangle=140, pctdistance=0.82, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Proportion of Each Severity')
plt.tight_layout()
plt.savefig('results/eda_01_severity_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('EDA - CVSS v3 Score Analysis', fontsize=14, fontweight='bold')
for sev, color in zip(SEVERITY_ORDER, PALETTE):
    subset = df_eda[df_eda['CVSS_v3_Severity']==sev]['CVSS_v3_Score'].dropna()
    axes[0].hist(subset, bins=20, alpha=0.6, color=color, label=sev, edgecolor='white')
axes[0].set_title('CVSS Score Distribution by Severity')
axes[0].set_xlabel('CVSS v3 Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(alpha=0.3)
box_data = [df_eda[df_eda['CVSS_v3_Severity']==s]['CVSS_v3_Score'].dropna() for s in SEVERITY_ORDER]
bp = axes[1].boxplot(box_data, labels=SEVERITY_ORDER, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_title('CVSS Score Boxplot per Severity')
axes[1].set_ylabel('CVSS v3 Score')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/eda_02_cvss_score_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EDA - Vulnerability Types (CWE) and Status', fontsize=14, fontweight='bold')
top_cwe = df_eda['CWE_ID'].value_counts().head(12)
axes[0].barh(top_cwe.index[::-1], top_cwe.values[::-1], color='#3498DB', edgecolor='white')
axes[0].set_title('Top 12 CWE Vulnerability Types')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('CWE ID')
axes[0].grid(axis='x', alpha=0.3)
status_counts = df_eda['Status'].value_counts()
axes[1].bar(status_counts.index, status_counts.values, color='#9B59B6', edgecolor='white')
axes[1].set_title('CVE Status Distribution')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/eda_03_cwe_status.png', bbox_inches='tight')
plt.show()

In [ ]:
df_eda2 = df_eda.copy()
df_eda2['pub_date'] = pd.to_datetime(df_eda2['Published_Date'], errors='coerce')
df_eda2['pub_month'] = df_eda2['pub_date'].dt.to_period('M')
monthly = df_eda2.groupby(['pub_month','CVSS_v3_Severity']).size().unstack(fill_value=0)
monthly.index = monthly.index.astype(str)
cols = [c for c in SEVERITY_ORDER if c in monthly.columns]
monthly = monthly[cols]
fig, ax = plt.subplots(figsize=(13, 5))
monthly.plot(kind='bar', stacked=True, ax=ax, color=PALETTE[:len(cols)],
             edgecolor='white', linewidth=0.5)
ax.set_title('CVE Publications per Month by Severity', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of CVEs')
ax.legend(title='Severity')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/eda_04_monthly_trend.png', bbox_inches='tight')
plt.show()

---
## 4. Feature Engineering (86 Features)
We engineer features from 8 groups:
- **Datetime** (8): publication timing, modification lag
- **CVSS v2** (1): legacy score (v3 excluded to avoid leakage)
- **Status** (6): one-hot encoded analysis status
- **CWE** (25): top-20 one-hot + 5 group flags
- **Description NLP** (15): length, word count, 11 security keyword flags
- **Affected Products** (4): product count, vendor count, open-source flag
- **References** (3): URL count, GitHub presence, advisory flag
- **TF-IDF** (30): top bigram/unigram terms from description text

> Note: CVSS v3 Score is excluded from features as it directly encodes the target label and would cause data leakage.

In [ ]:
def engineer_features(df):
    d = df.copy()
    for col in ['Published_Date', 'Last_Modified']:
        d[col] = pd.to_datetime(d[col], errors='coerce')

    # Datetime features
    d['pub_year']       = d['Published_Date'].dt.year
    d['pub_month_num']  = d['Published_Date'].dt.month
    d['pub_day']        = d['Published_Date'].dt.day
    d['pub_hour']       = d['Published_Date'].dt.hour
    d['pub_dayofweek']  = d['Published_Date'].dt.dayofweek
    d['pub_quarter']    = d['Published_Date'].dt.quarter
    d['days_since_pub'] = (datetime(2026,4,21) - d['Published_Date']).dt.days.clip(0, 3650)
    d['mod_lag_days']   = (d['Last_Modified'] - d['Published_Date']).dt.days.clip(0, 3650)

    # CVSS v2 only (v3 excluded - would leak the target)
    d['cvss_v2_score']  = pd.to_numeric(d['CVSS_v2_Score'], errors='coerce')

    # Status one-hot
    statuses = ['Analyzed', 'Awaiting Analysis', 'Undergoing Analysis',
                'Received', 'Modified', 'Deferred']
    for s in statuses:
        d[f'status_{s.lower().replace(" ","_")}'] = (d['Status'] == s).astype(int)

    # CWE one-hot (top 20)
    top_cwes = ['CWE-79','CWE-89','CWE-862','CWE-22','CWE-74','CWE-918',
                'CWE-787','CWE-78','CWE-863','CWE-416','CWE-77','CWE-119',
                'CWE-20','CWE-125','CWE-200','CWE-639','CWE-94','CWE-284',
                'CWE-352','CWE-502']
    for cwe in top_cwes:
        d[f'cwe_{cwe.replace("-","_")}'] = d['CWE_ID'].str.contains(cwe, na=False).astype(int)

    # CWE group flags
    d['cwe_is_injection'] = d['CWE_ID'].isin(['CWE-89','CWE-78','CWE-77','CWE-74','CWE-94']).astype(int)
    d['cwe_is_memory']    = d['CWE_ID'].isin(['CWE-787','CWE-119','CWE-125','CWE-416']).astype(int)
    d['cwe_is_auth']      = d['CWE_ID'].isin(['CWE-862','CWE-863','CWE-284','CWE-639']).astype(int)
    d['cwe_is_ssrf']      = d['CWE_ID'].isin(['CWE-918']).astype(int)
    d['cwe_is_known']     = d['CWE_ID'].notna().astype(int)

    # Description NLP
    desc = d['Description'].fillna('')
    d['desc_len']          = desc.str.len()
    d['desc_word_count']   = desc.str.split().str.len()
    d['desc_sentence_cnt'] = desc.str.count(r'[.!?]')
    d['desc_avg_word_len'] = desc.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)

    keywords = {
        'kw_remote':     r'remote|unauthenticated|network',
        'kw_execute':    r'execut|remote code|rce|command inject',
        'kw_overflow':   r'overflow|buffer|heap|stack',
        'kw_injection':  r'inject|sql|xss|cross.site',
        'kw_privilege':  r'privilege|escalat|root|admin',
        'kw_bypass':     r'bypass|circumvent|evad',
        'kw_disclosure': r'disclos|leak|exposure|information',
        'kw_dos':        r'denial.of.service|dos|crash|unavailab',
        'kw_auth':       r'authenticat|authoriz|permission|access control',
        'kw_ssrf':       r'ssrf|server.side|request forgery',
        'kw_traversal':  r'traversal|path|directory',
    }
    for feat, pattern in keywords.items():
        d[feat] = desc.str.lower().str.contains(pattern, regex=True).astype(int)

    # Affected Products
    prod = d['Affected_Products'].fillna('')
    d['prod_count']         = prod.str.count(r'cpe:').clip(0, 50)
    d['prod_has_any']       = (prod != '').astype(int)
    d['prod_vendor_cnt']    = prod.apply(lambda x: len(set(re.findall(r'cpe:2\.3:.:([^:]+):', x))))
    d['prod_is_opensource'] = prod.str.contains('github|apache|linux|wordpress', case=False, na=False).astype(int)

    # Reference URLs
    refs = d['Reference_URLs'].fillna('')
    d['ref_count']        = refs.str.count(';') + (refs != '').astype(int)
    d['ref_has_github']   = refs.str.contains('github', case=False, na=False).astype(int)
    d['ref_has_advisory'] = refs.str.contains('advisory|GHSA|CVE', na=False).astype(int)

    return d


df_feat = engineer_features(df_raw)
print(f'Feature engineering complete. Shape: {df_feat.shape}')

In [ ]:
# Filter to labelled rows only
df_model = df_feat.dropna(subset=['CVSS_v3_Severity']).copy()
df_model = df_model[df_model['CVSS_v3_Severity'].isin(SEVERITY_ORDER)].reset_index(drop=True)

# TF-IDF on description text
tfidf = TfidfVectorizer(max_features=30, stop_words='english', ngram_range=(1,2), min_df=5)
tfidf_matrix = tfidf.fit_transform(df_model['Description'].fillna(''))
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                         columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()])
df_model = pd.concat([df_model, tfidf_df], axis=1)
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Define feature columns (exclude raw/target columns)
EXCLUDE = ['CVE_ID','Published_Date','Last_Modified','Status','Description',
           'CVSS_v3_Score','CVSS_v3_Severity','CVSS_v2_Score','CWE_ID',
           'Affected_Products','Reference_URLs','pub_month']
feature_cols = [c for c in df_model.columns
                if c not in EXCLUDE and df_model[c].dtype in [np.float64, np.int64, float, int]]

X = df_model[feature_cols].copy()
le = LabelEncoder()
le.fit(SEVERITY_ORDER)
y = le.transform(df_model['CVSS_v3_Severity'])

# Impute and scale
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

# Save preprocessors
with open('models/label_encoder.pkl', 'wb') as f: pickle.dump(le, f)
with open('models/imputer.pkl', 'wb') as f: pickle.dump(imputer, f)
with open('models/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
with open('models/feature_cols.pkl', 'wb') as f: pickle.dump(feature_cols, f)

# Correlation heatmap
corr_mat = X_imp[feature_cols[:20]].corr()
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr_mat, annot=False, cmap='coolwarm', center=0, linewidths=0.3, ax=ax)
ax.set_title('Feature Correlation Heatmap (Top 20 Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/eda_05_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f'Total features : {len(feature_cols)}')
print(f'Training rows  : {X_train.shape[0]:,}')
print(f'Test rows      : {X_test.shape[0]:,}')
print(f'Classes        : {le.classes_}')

---
## 5. Model Training
Four classifiers are trained and compared:
| Algorithm | Justification |
|---|---|
| **Random Forest** | Ensemble of trees; robust, handles mixed features, gives feature importances |
| **XGBoost** | Gradient boosting; state-of-the-art on tabular data, handles class imbalance |
| **Logistic Regression** | Interpretable linear baseline; fast convergence |
| **K-Nearest Neighbours** | Non-parametric instance-based learner; good distance comparison |

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_split=5,
        class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', solver='lbfgs', random_state=42),
    'K-Nearest Neighbours': KNeighborsClassifier(
        n_neighbors=9, weights='distance', n_jobs=-1),
}

results = {}
cv_scores = {}
colors_m = ['#2ECC71', '#E74C3C', '#3498DB', '#F39C12']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = list(models.keys())

for name, model in models.items():
    print(f'Training {name} ...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cv   = cross_val_score(model, X_scaled, y, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_scores[name] = cv
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'cv_mean': cv.mean(), 'cv_std': cv.std()
    }
    safe = name.lower().replace(' ', '_').replace('-', '_')
    with open(f'models/{safe}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f'  Acc={acc:.4f}  F1={f1:.4f}  CV={cv.mean():.4f}+/-{cv.std():.4f}  -> saved {safe}.pkl')

# Summary table
summary = pd.DataFrame([{
    'Model': n, 'Accuracy': r['accuracy'], 'Precision': r['precision'],
    'Recall': r['recall'], 'F1 Score': r['f1'],
    'CV Mean': r['cv_mean'], 'CV Std': r['cv_std']}
    for n, r in results.items()]).set_index('Model').sort_values('F1 Score', ascending=False)
summary.to_csv('results/metrics_summary.csv')
print('\n=== Performance Summary ===')
print(summary.round(4))

---
## 6. Results Visualisation

In [ ]:
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(metrics_list))
bar_width = 0.18
fig, ax = plt.subplots(figsize=(13, 6))
for i, (mname, color) in enumerate(zip(model_names, colors_m)):
    r = results[mname]
    vals = [r['accuracy'], r['precision'], r['recall'], r['f1']]
    bars = ax.bar(x + i*bar_width, vals, bar_width, label=mname, color=color, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax.set_xticks(x + bar_width*1.5)
ax.set_xticklabels(metrics_list, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison - All Metrics', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim(0, 1.08)
ax.grid(axis='y', alpha=0.3)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('results/result_01_metrics_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
class_labels = le.classes_
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices - All Models', fontsize=15, fontweight='bold', y=1.01)
for ax, (name, r) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_labels, yticklabels=class_labels,
                linewidths=0.5, cbar=False, annot_kws={'size':10, 'weight':'bold'})
    ax.set_title(f'{name}\nAcc={r["accuracy"]:.3f}  F1={r["f1"]:.3f}', fontsize=11)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('results/result_02_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
cv_data = [cv_scores[m] for m in model_names]
bp = ax.boxplot(cv_data, labels=model_names, patch_artist=True,
                medianprops=dict(color='black', linewidth=2.5))
for patch, color in zip(bp['boxes'], colors_m):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
for i, m in enumerate(cv_data):
    ax.scatter(i+1, np.mean(m), color='black', zorder=5, s=60)
    ax.text(i+1, np.mean(m)+0.003, f'{np.mean(m):.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_title('5-Fold Cross-Validation Accuracy Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(model_names, rotation=15)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0.3, 1.02)
plt.tight_layout()
plt.savefig('results/result_03_cross_validation.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6), sharey=True)
fig.suptitle('Per-Class F1 Score per Model', fontsize=13, fontweight='bold')
for ax, (name, r), mcolor in zip(axes, results.items(), colors_m):
    report = classification_report(y_test, r['y_pred'],
                                    target_names=class_labels, output_dict=True)
    class_f1 = [report[c]['f1-score'] for c in class_labels]
    bars = ax.bar(class_labels, class_f1, color=PALETTE, edgecolor='white')
    for bar, v in zip(bars, class_f1):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('F1 Score')
plt.tight_layout()
plt.savefig('results/result_04_per_class_f1.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top 20 Feature Importances', fontsize=13, fontweight='bold')
for ax, mname, color in zip(axes, ['Random Forest', 'XGBoost'], ['#2ECC71', '#E74C3C']):
    importances = results[mname]['model'].feature_importances_
    fi = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(20)
    ax.barh(fi.index[::-1], fi.values[::-1], color=color, edgecolor='white', alpha=0.85)
    ax.set_title(mname, fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('results/result_05_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
y_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Multiclass ROC Curves (One-vs-Rest)', fontsize=13, fontweight='bold')
for ax, mname in zip(axes, ['Random Forest', 'XGBoost']):
    y_score = results[mname]['model'].predict_proba(X_test)
    for i, (sev, color) in enumerate(zip(SEVERITY_ORDER, PALETTE)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{sev} (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.set_title(mname, fontsize=11, fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/result_06_roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
metric_labels_r = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'CV Mean']
N = len(metric_labels_r)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_facecolor('#f8f9fa')
for (mname, r), color in zip(results.items(), colors_m):
    vals = [r['accuracy'], r['precision'], r['recall'], r['f1'], r['cv_mean']]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=mname, color=color)
    ax.fill(angles, vals, alpha=0.07, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels_r, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.grid(color='grey', linestyle='--', alpha=0.4)
ax.set_title('Model Performance Radar Chart', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('results/result_07_radar_chart.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 70)
for name, r in results.items():
    print(f'\n### {name} ###')
    print(classification_report(y_test, r['y_pred'], target_names=le.classes_, zero_division=0))
    print('-' * 70)

---
## 7. Download All Files

In [ ]:
import shutil
print('=== Saved Model Files ===')
for f in sorted(os.listdir('models')):
    print(f'  models/{f}  {os.path.getsize("models/"+f)/1024:.1f} KB')
print('\n=== Saved Result Files ===')
for f in sorted(os.listdir('results')):
    print(f'  results/{f}  {os.path.getsize("results/"+f)/1024:.1f} KB')

shutil.make_archive('CMP7239_output', 'zip', '.', '.')
try:
    from google.colab import files
    files.download('CMP7239_output.zip')
except Exception:
    print('File saved as CMP7239_output.zip')

---
## 8. Conclusion

### Problem Statement
This project applies machine learning to automatically classify the severity of CVEs from the NVD feed, reducing manual analyst effort in vulnerability triage and patch prioritisation.

### Dataset
- **Source:** National Vulnerability Database (NVD) - Last 30 Days
- **Size:** 6,601 CVEs x 11 raw columns -> 5,945 labelled samples after cleaning
- **Target:** CVSS v3 Severity (CRITICAL / HIGH / MEDIUM / LOW)

### Feature Engineering Summary
86 features engineered from 11 raw columns across datetime, CWE taxonomy, NLP keyword flags, TF-IDF, product metadata, and reference URL signals. CVSS v3 Score deliberately excluded to prevent data leakage.

### Key Findings
- **Random Forest** achieved the best weighted F1 of 0.670, followed closely by XGBoost at 0.662
- CRITICAL and LOW classes are underrepresented and harder to classify correctly
- CWE type, description keyword flags, and TF-IDF terms are the most predictive features
- Logistic Regression underperformed due to non-linear relationships in the data

### Limitations
- Class imbalance (CRITICAL=10%, LOW=4%) affects minority-class recall
- TF-IDF loses semantic meaning compared to transformer-based embeddings
- Missing CVSS v2 scores (~89%) reduce score-based feature utility

### Recommendations
- Apply SMOTE oversampling to address class imbalance
- Replace TF-IDF with SecBERT embeddings for richer text representations
- Deploy best model (Random Forest) as a real-time CVE triage API